# 密钥管理协议设计与实现 — 自学课程

**分类**：密码协议设计专项

**内容简介**：涵盖DH/ECDH/KDC/STS协议，详解原理、设计与Python实现



## 学习目标

- 用“消息序列”描述协议
- 写出最小可运行模拟
- 识别重放/中间人/身份绑定等常见风险
- 通过小实验理解修补策略



## 工具箱（标准库）

- Hash：$H(m)=\mathrm{SHA256}(m)$
- MAC：$t=\mathrm{HMAC}_k(m)$
- Nonce：$n\leftarrow\{0,1\}^{\lambda}$



In [ ]:
# Step 0：基础密码工具
import hashlib, hmac, secrets, time
def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()
def hmac_sha256(key: bytes, msg: bytes) -> bytes:
    return hmac.new(key, msg, hashlib.sha256).digest()
def randbytes(n: int) -> bytes:
    return secrets.token_bytes(n)

print(len(randbytes(16)))
# 预期输出：16



## 自检小测

1) 为什么协议里经常需要 nonce？
2) 为什么建议用 hmac.compare_digest 做比较？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



# Step 1：PSK 场景与会话密钥派生

我们用预共享密钥 $k_{psk}$ 与双方随机数派生会话密钥：

$$k_s = H(k_{psk}\|n_A\|n_B)$$



In [ ]:
# Step 1：派生会话密钥
def kdf(psk: bytes, nA: bytes, nB: bytes) -> bytes:
    return sha256(psk + b"|" + nA + b"|" + nB)

psk=b"demo-psk"
nA=randbytes(16); nB=randbytes(16)
kA=kdf(psk,nA,nB); kB=kdf(psk,nA,nB)
print(kA==kB)
# 预期输出：True



# Step 2：双向确认（Key Confirmation）

双方用 HMAC 确认“我们确实得到了同一个会话密钥”。



In [ ]:
# Step 2：确认标签
def mac(k: bytes, label: str, nA: bytes, nB: bytes) -> bytes:
    return hmac_sha256(k, label.encode()+b"|" + nA + b"|" + nB)

tA=mac(kA,"A->B",nA,nB)
tB=mac(kB,"B->A",nA,nB)
print(hmac.compare_digest(tA, mac(kB,"A->B",nA,nB)))
print(hmac.compare_digest(tB, mac(kA,"B->A",nA,nB)))
# 预期输出：
# True
# True



# Step 3：重放攻击模拟

如果攻击者把旧的 $(n_A,n_B,t)$ 重放，系统可能错误接受。我们用“已见 nonce 集合”阻止重放。



In [ ]:
# Step 3：重放防护示例
seen=set()
def accept_handshake(nA: bytes, nB: bytes):
    sid = sha256(nA+b"|"+nB)  # 会话标识
    if sid in seen:
        return False
    seen.add(sid)
    return True

print(accept_handshake(nA,nB))
print(accept_handshake(nA,nB))
# 预期输出：
# True
# False



# Step 4：身份绑定（防 Unknown Key-Share）

把身份写入 KDF：

$$k_s=H(k_{psk}\|A\|B\|n_A\|n_B)$$



In [ ]:
# Step 4：身份绑定
def kdf_bound(psk: bytes, A: str, B: str, nA: bytes, nB: bytes) -> bytes:
    return sha256(psk + A.encode()+b"|" + B.encode()+b"|" + nA + b"|" + nB)

k1=kdf_bound(psk,"Alice","Bob",nA,nB)
k2=kdf_bound(psk,"Bob","Alice",nA,nB)
print(k1==k2)
# 预期输出：False



## 练习

1) 把时间戳加入 sid，并实现简单的过期窗口检查。
2) 模拟弱随机数：把 nA 设为全 0，讨论风险。
3) 思考：如果 psk 泄漏，攻击者能做什么？如何减轻影响？


## 总结与延伸

- 协议安全分析离不开“对手模型”：攻击者能监听？能篡改？能冒充？
- 实现时优先使用成熟库与标准（如 TLS）；本课程实现仅用于学习。

> 下一步：学习形式化验证、协议逻辑、以及常见真实协议的攻击案例。



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”



## 自检小测

1) 如果攻击者能选择明文并看到密文，这叫哪种攻击模型？
2) 如果攻击者只看到密文，常见的第一步是什么？

> 建议先不看代码答案，自己写出推理或在纸上算一遍。



## 实用检查清单

- [ ] 输入规范化是否一致？
- [ ] 密钥边界情况是否处理？（空 key、重复字母、不可逆 a…）
- [ ] 是否写了最小测试？
- [ ] 是否能解释每一步的安全直觉？
- [ ] 输出是否可复现（随机数是否可控）？



In [ ]:
# 轻量测试模板：你可以把它复制到任何课程里
def roundtrip_test(encrypt, decrypt, samples, key):
    for s in samples:
        c = encrypt(s, key)
        p = decrypt(c, key)
        assert p == normalize(s), (s, c, p)
    print("roundtrip ok")

# 示例（如果你的课程定义了 caesar_*，可取消注释运行）
# roundtrip_test(lambda m,k: caesar_encrypt_text(m,k),
#               lambda c,k: caesar_decrypt_text(c,k),
#               ["HELLO", "ATTACK AT DAWN", "Sphinx of black quartz"], 3)



## Mini Project（可选）

选择一个小项目完成并写 5~10 行总结：

1) **自动破译器**：输入密文，输出 top-5 候选明文与参数
2) **评分器对比**：实现 3 种评分器，比较命中率与误判
3) **组合密码实验**：把两种古典方法组合，尝试设计破译策略

> 重点：写清楚你做了什么、为什么这样做、观察到了什么。



## 常见问题

- Q：为什么这些方法不安全？
  - A：密钥空间小、结构线性、统计特征泄漏。
- Q：那我该用什么？
  - A：真实场景使用标准化现代算法与协议（如 AES、TLS），不要自造。



## 延伸阅读方向（不依赖外部库也能做）

- 实现更强的统计评分：bigram/trigram、卡方检验
- 加入简单的单元测试框架（用 assert 即可）
- 写一个命令行交互（如果环境允许）
- 把算法封装成类/模块并写文档字符串



## 术语速记

- 明文/密文：加密前/后的文本
- 密钥：控制加密变换的秘密参数
- 攻击模型：攻击者能获得/能做什么（监听、篡改、查询…）
- 正确性：解密能还原明文
- 安全性：在给定攻击模型下“难以被破坏”

